### Initializing instrumentation

In [1]:
# import packages
from Equipments import BNC575, Weeder, SR400, MCBOX, Andor
from Equipments.MCBOX import generate_test_voltages
from DataProcessing.Plot import PlotSaver
import numpy as np
import pyvisa
import time
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

In [2]:
ps = PlotSaver("X:/migratedData/Rydberg_QIS/data")

In [3]:
rm = pyvisa.ResourceManager()
print(rm.list_resources())

('TCPIP0::K-34465A-15333::hislip0::INSTR', 'TCPIP0::K-34465A-15333::inst0::INSTR', 'ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'ASRL6::INSTR', 'ASRL7::INSTR', 'GPIB0::9::INSTR', 'TCPIP0::169.254.110.10::inst0::INSTR', 'USB0::0x1313::0x807B::241023118::0::INSTR')


In [4]:
bnc575 = BNC575.BNC575("GPIB0::9::INSTR")

In [ ]:
sr400 = SR400.SR400("GPIB0::23::INSTR", timeout = 1000)

In [ ]:
weeder = Weeder.Weeder("ASRL1::INSTR")

In [ ]:
mcbox = MCBOX.MCBOX(find_device = "USB-3114")

### Defining the pulse sequence

In [ ]:
pulse_arrangement = [["A", 900e-6, 0, "TTL"],              # MOT
                     [["B", 10e-6, 5e-6, 15],              # 780 EXC (20dB attenuator connected)
                      ["C", 10e-6, 5e-6, "TTL"]],          # 480 EXC
                     ["D", 20e-6, 0, 8],                   # DEI switch
                    ]
pulse_arrangement = [
                     ["D", 20e-6, 0, 8],                   # DEI switch
                    ]
T = 1000e-6

cycle_number = 30

# photon counter gate width and delay
gate_width = 15e-6
gate_delay = 3e-6

# stepper motor slope (for blue laser)
freq_factor = 129 * 2 # kHz

notes = {"gate_width": gate_width, "gate_delay": gate_delay}

In [ ]:
sr400.counter_set(count_mode = "INDEPENDENT", count_preset = 100e-6, count_period_num = cycle_number, 
                  dwell = 0, sourceA = "INPUT1", sourceB = "INPUT2", gate_A_mode = "FIXED",
                  gate_A_delay = gate_delay, gate_A_width = gate_width, gate_B_mode = "CW")

In [ ]:
notes["bnc575 T0"] = bnc575.clock_set(period = T, mode = "CONTINUOUS")
notes["bnc575 pulses"] = bnc575.pulse_sequence_setup(pulse_arrangement)

In [ ]:
bnc575.start_pulses()

### Search for a Rydberg resonance

In [ ]:
weeder.position(header = "A", query = True)

In [ ]:
# begin stepper motor at initial position 1000, then move it out to increase the frequency
weeder.move("A", position = 2800, progress = True)

In [ ]:
# used to move motor incrementally to zero in on the location of a resonance
weeder.advance(header = "A", num_step = 1)

### Generating Stark maps 

#### 1D map

In [ ]:
center = 2900
stride = 2
half_width = 200

# scan list gneration
x_list = np.arange(2* half_width/stride)

In [ ]:
# everytime go further and go back to release the band tension and gear margin
start_position = center - half_width
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

results = []
s = time.perf_counter()
for idx in x_list:
    print(f"Now checking step: {idx}", end = "\r", flush = True)
    sr400.count_reset()
    result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
    results.append(result)
    for i in range(stride):
        weeder.step("A", "+")
    time.sleep(0.1)
    
e = time.perf_counter()
print('Time elapsed: {} seconds'.format(e-s))

In [ ]:
# plot results
ps.Plot2D(x = x_list*stride*freq_factor, y = None, Z = results,
          xlabel = "480 laser frequency (kHz)", zlabel = "Average number of counts", notes = notes)

#### 2D maps (small-voltage range)

**Method 1: Frequency scan inside of voltage scan (not using)**

**Method 2: Voltage scan inside of frequency scan (preferred)**

#### 2D Stark maps (full voltage range)

In [ ]:
center = 2480  # center point of frequency scan
stride = 2  # step size
half_width = 200  # step range

# scan list gneration
s_list = np.arange(2*half_width/stride)

# voltage setpoints
vset = np.array([0.2,-0.1,0.6])

num_voltages = 51  # number of voltages to test

# voltage arrays for each axis
vx_list = np.linspace(-10.0,10.0,num_voltages) + vset[0]
vy_list = np.linspace(-10.0,10.0,num_voltages) + vset[1]
vz_list = np.linspace(-10.0,10.0,num_voltages) + vset[2]

v_list = np.vstack((vx_list,vy_list,vz_list))

# MC channels for each electrode pair
electrode_pair = [[10,11],[6,7],[0,1]]

In [ ]:
# initialize electrode voltages
mcbox.set_voltage_1chan(board_num = 0, channel = 10, voltage = vset[0], bipolar = True, display = True)
mcbox.set_voltage_1chan(board_num = 0, channel = 11, voltage = -vset[0], bipolar = True, display = True)

mcbox.set_voltage_1chan(board_num = 0, channel = 6, voltage = vset[1], bipolar = True, display = True)
mcbox.set_voltage_1chan(board_num = 0, channel = 7, voltage = -vset[1], bipolar = True, display = True)

mcbox.set_voltage_1chan(board_num = 0, channel = 0, voltage = 0, bipolar = True, display = True)
mcbox.set_voltage_1chan(board_num = 0, channel = 1, voltage = -vset[2], bipolar = True, display = True)

In [ ]:
# voltage scan inside a frequency scan
start_position = center - half_width

stark_map = []

# electrode pair to check; i = 0,1,2 for x,y,z
i = 2

test_voltages = generate_test_voltages(v_list[i])
print('Voltages to test: \n {}'.format(test_voltages))

print('Now checking axis {}'.format(i))
weeder.move(header = "A", position = start_position - 100)
weeder.move(header = "A", position = start_position)

s = time.perf_counter()
#print('start:',s)

for s in s_list:
    for j in range(stride):
            weeder.step("A", "+")
    print(f"Now checking step: {s}", end = "\r", flush = True)
    
    results = []
    for v_s in test_voltages:
        # scan voltage
        if electrode_pair[i][0] == 0:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = 0, bipolar = True, display = False)
        else:
            mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][0], voltage = v_s, bipolar = True, display = False)
        mcbox.set_voltage_1chan(board_num = 0, channel = electrode_pair[i][1], voltage = -v_s, bipolar = True, display = False)
        time.sleep(0.01)
        sr400.count_reset()
        result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
        results.append(result)
    
    stark_map.append(results)

e = time.perf_counter()
#print('end:',e)
print('Time elapsed: {} seconds'.format(e-s))

In [ ]:
# plot results
ps.Plot3D(x = test_voltages, y = s_list*stride*freq_factor, Z = np.array(stark_map).T,
          xlabel = f"Voltage {i} (V)", ylabel = "480 laser frequency (kHz)", zlabel = "Average number of counts", notes = notes)

In [ ]:
# frequency scan inside a voltage scan
start_position = center - half_width

stark_map = []

#s = time.perf_counter()
for v in test_voltages:
    mcbox.set_voltage_1chan(board_num = 0, channel = 10, voltage = v, bipolar = True)
    mcbox.set_voltage_1chan(board_num = 0, channel = 11, voltage = -v, bipolar = True)
    
    # everytime go further and go back to release the band tension and gear margin
    weeder.move(header = "A", position = start_position-100)
    weeder.move(header = "A", position = start_position)
    results = []
    for idx in s_list:
        print(f"Now checking step: {idx}", end = "\r", flush=True)
        sr400.count_reset()
        result = np.mean(sr400.read_entire_counts(counter = "A", length = cycle_number))
        results.append(result)
        for i in range(stride):
            weeder.step("A", "+")
        time.sleep(0.1)
    stark_map.append(results)

#e = time.perf_counter()
#print('Time elapsed: {} seconds'.format(e-s))

In [ ]:
# plot results
ps.Plot3D(x = test_voltages, y = s_list*stride*freq_factor, Z = np.array(stark_map),
          xlabel = f"Voltage {i} (V)", ylabel = "480 laser frequency (kHz)", zlabel = "Average number of counts", notes = notes)

### Peak finding and curve fitting (old code; needs to be updated)

### Shutting down instrumentation

In [ ]:
# reset voltages to setpoint values
mcbox.set_voltage_Nchan(board_num = 0, channels = [0,1,6,7,10,11], voltages = [0,-vset[2],vset[1],-vset[1],vset[0],-vset[0]], bipolar = True)

In [ ]:
# save current stepper motor position
weeder.save(header = "A", query = True)

In [ ]:
# OPTIONAL: disables pulses from BNC box (can also be done by pressing RUN/STOP button or individual channel buttons on device)
bnc575.disable_all()

In [ ]:
# OPTIONAL: disarms the pulse generator (can also be done by pressing RUN/STOP button on device)
bnc575.disarm_all()

In [ ]:
# OPTIONAL: used to reset photon counter (can also be done by pressing STOP button on device)
sr400.count_reset()

In [ ]:
# disconnect from BNC box
bnc575.pyvisa.close()

In [ ]:
# disconnect from SRS photon counter
sr400.pyvisa.close()